# 제조 음성 ASR 화자 일반화 회복 v3 — A100 + 고용량 RAM

기존 speaker-held-out Test v2에서 **Recall 82.33%, CER 17.35%, WER 36.64%**로 품질 gate를 통과하지 못한 원인을 개선하는 개발용 노트북입니다. Train/Validation만 새로 만들고, 모델·학습량은 새 Validation에서 선택합니다. 이미 확인한 Test v1/v2를 보며 설정을 다시 조정하지 않습니다.

> 합성 음성은 기술 검증 자료입니다. 실제 작업자·공장 소음에 대한 생산 적용 주장은 승인된 shadow test와 사람 검수 뒤에만 가능합니다.

In [ ]:
# GitHub 작업 브랜치와 Colab·Google Drive의 재사용 경로를 지정합니다.
GITHUB_REPO_URL = 'https://github.com/Pronesis9758/aias-specialist-asr.git'
GITHUB_BRANCH = 'codex/whisper-benchmark-quantization'
PROJECT_DIR = '/content/AIAS'
DRIVE_ROOT = '/content/drive/MyDrive/AI_Specialist_ASR_Project'

# 데이터 생성은 최초 한 번만 True로 바꾸며, 완성 WAV는 Drive에서 재사용합니다.
GENERATE_GENERALIZATION_AUDIO = False
# None이면 학습하지 않습니다. pilot-5h → target-10h → extended-15h 순서로만 확장합니다.
RUN_LORA_THROUGH_STAGE = None  # 예: 'pilot-5h'


## 1. A100·고용량 RAM·Drive 확인

A100 40GB 이상이 아니면 LoRA CLI가 비용 작업 전에 중단됩니다. RAM 수치도 증빙으로 출력합니다.

In [ ]:
!nvidia-smi
!free -h
from pathlib import Path
from google.colab import drive

# Colab 세션이 Drive를 아직 마운트하지 않은 경우에만 Google 승인을 요청합니다.
drive_root = Path('/content/drive/MyDrive')
if not drive_root.is_dir():
    drive.mount('/content/drive')
if not drive_root.is_dir():
    raise RuntimeError('Google Drive 연결을 확인할 수 없습니다.')


## 2. 코드 동기화·의존성 설치

실행 commit과 Python/GPU 환경을 로그에 남기고 모든 CLI 로그를 Colab에 즉시 표시합니다.

In [ ]:
import os
import subprocess
import sys

# 저장소가 없으면 복제하고, 있으면 작업 브랜치만 fast-forward 갱신합니다.
if not Path(PROJECT_DIR).exists():
    subprocess.run([
        'git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch',
        GITHUB_REPO_URL, PROJECT_DIR,
    ], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run([
        'git', '-C', PROJECT_DIR, 'merge', '--ff-only', f'origin/{GITHUB_BRANCH}',
    ], check=True)
os.chdir(PROJECT_DIR)
print('Git commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version)


In [ ]:
# Colab 사전 설치 패키지와 충돌할 수 있는 선택 UI 패키지를 제거하고 학습·TTS 의존성을 설치합니다.
%pip uninstall -y torchao gradio gradio-client
%pip install -q -e '.[train]' 'transformers>=4.46,<5' 'peft>=0.14,<0.19' 'edge-tts>=7,<8'

# 현재 Python으로 AIAS CLI를 실행해 장시간 작업 로그와 실패 원인을 즉시 표시합니다.
def run_aias(*args):
    command = [sys.executable, '-m', 'aias_specialist.cli', *args]
    print('\nRunning:', ' '.join(command), flush=True)
    subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})


## 3. 일반화 강화 데이터 계획·생성

Train 9,000건(15h), Validation 1,200건(2h), 서로 겹치지 않는 96개 음향 프로필을 사용합니다. v2 실패 용어와 밝고 빠른 발화를 집중 보강하며 Test는 포함하지 않습니다.

In [ ]:
import json
import pandas as pd

DATA_SPEC = 'configs/data/synthetic_manufacturing_generalization_v3.yaml'
run_aias('plan-synthetic-dataset', '--spec', DATA_SPEC)
data_root = Path(DRIVE_ROOT) / 'data/synthetic/manufacturing_generalization_v3'
plan = pd.read_csv(data_root / 'manifest.plan.csv')
display(plan.groupby('split').size().rename('samples'))
display(plan.groupby('split')['speaker_id'].nunique().rename('acoustic_profiles'))
display(json.loads((data_root / 'plan_summary.json').read_text(encoding='utf-8')))

# 10,200개 TTS 생성은 resumable이며 이미 완료한 WAV는 건너뜁니다.
if GENERATE_GENERALIZATION_AUDIO:
    # Edge TTS 안정성 실측을 반영해 동시 요청을 4개로 제한합니다.
    # 이미 생성된 WAV는 자동으로 재사용합니다.
    run_aias('synthesize-dataset', '--spec', DATA_SPEC, '--concurrency', '4')
else:
    print('음성 생성을 건너뜁니다. 최초 생성 시에만 True로 변경하세요.')
manifest_path = data_root / 'manifest.csv'
print('Final manifest exists:', manifest_path.exists(), manifest_path)


## 4. Turbo·Large-v3 누적 LoRA 학습곡선

먼저 `pilot-5h`까지만 실행합니다. Recall 증가가 남아 있거나 CER/WER가 목표에 미달하면 `target-10h`, 이후 필요할 때만 `extended-15h`로 확장합니다. 완료된 앞 단계 worker는 다시 학습하지 않습니다.

In [ ]:
LORA_SPEC = 'configs/training/synthetic_manufacturing_lora_a100_generalization_v3.yaml'
allowed_stages = {None, 'pilot-5h', 'target-10h', 'extended-15h'}
if RUN_LORA_THROUGH_STAGE not in allowed_stages:
    raise ValueError(f'RUN_LORA_THROUGH_STAGE는 {allowed_stages} 중 하나여야 합니다.')
if RUN_LORA_THROUGH_STAGE is not None:
    if not manifest_path.exists():
        raise FileNotFoundError('3단계에서 합성 WAV와 최종 manifest를 먼저 생성하세요.')
    run_aias(
        'lora-learning-curve', '--spec', LORA_SPEC,
        '--max-stage', RUN_LORA_THROUGH_STAGE,
    )
else:
    print('LoRA 학습을 건너뜁니다. 최초에는 pilot-5h를 지정하세요.')

lora_dir = (
    Path(DRIVE_ROOT)
    / 'artifacts/training/synthetic-manufacturing-lora-a100-generalization-v3'
)
curve_path = lora_dir / 'lora_learning_curve.csv'
if curve_path.exists():
    curve = pd.read_csv(curve_path)
    display(curve[[
        'stage_id', 'model_id', 'train_samples', 'domain_term_recall', 'cer', 'wer', 'rank'
    ]])
    curve['quality_gate_pass'] = (
        curve['domain_term_recall'].ge(0.85)
        & curve['cer'].le(0.07)
        & curve['wer'].le(0.15)
    )
    display(curve[['stage_id', 'model_id', 'quality_gate_pass']])


## 5. 선택·다음 평가 규칙

모델과 학습량은 위 새 Validation에서만 고정합니다. 그 뒤 디코딩·IR/NN·양자화도 별도 Validation gate를 통과한 경우에만 채택하고, **신규 화자·신규 문장·신규 WAV의 frozen Test v3를 새로 만든 뒤 딱 한 번** 평가합니다. v1/v2 결과는 이 재튜닝에 사용하지 않습니다.